In [ ]:
## In this example, we will study output data frame from pandora.py configuration
#### 1. Opening each data frame and check structure
#### 2. Collect POT and scale factor to the target POT
#### 3. Merge evtdf and mcnudf for further study
#### 4. Draw some plots for each slice and for each pfp


import os
import sys
import lmfit
import numpy as np
import math
import uproot as uproot
import pickle
import pandas as pd
import gc

import matplotlib.pyplot as plt
import matplotlib.colors
from matplotlib.colors import LinearSegmentedColormap
from matplotlib import ticker
from matplotlib.ticker import (AutoMinorLocator, MultipleLocator)
from matplotlib import gridspec

# Absolute path to cafpyana directory
print('Importing cafpyana utils...')
cafpyana_root = "/home/lpelegri/cafpyana"
# Add CAFpyana to the Python search path -- this will allow you to import cafpyana modules
sys.path.insert(0, cafpyana_root)

from analysis_village.unfolding.wienersvd import *
from analysis_village.unfolding.unfolding_inputs import *
from analysis_village.cc1pi.var_configs import *



from analysis_village.cc1pi.HelperFunctions import HelperFunctions
from analysis_village.cc1pi.Constants import CTE as CTE
from analysis_village.cc1pi.CutMasks import CutMasks
from analysis_village.cc1pi.CutMasks.MaskUtils import *
from analysis_village.cc1pi.DataFrameUtils import DFCleaning
from analysis_village.cc1pi.DataFrameUtils.DFLoading import *
from analysis_village.cc1pi.GraphUtils.GraphsUtils import *
from analysis_village.cc1pi.GraphUtils.Utils import *

# import this repo's classes
import pyanalib.pandas_helpers as ph
import pyanalib.split_df_helpers as splh
import pyanalib.stat_helpers as sh

from pyanalib.split_df_helpers import *
from analysis_village.cc1pi.systematics.final_variable_configs import VariableConfig
from analysis_village.cc1pi.systematics.utils import *
from analysis_village.cc1pi.systematics.constants import *
from pyanalib.covariance import *
from analysis_village.cc1pi.DataFrameUtils.DFLoading import *

np.seterr(divide='ignore', invalid='ignore', over='ignore')

# Load DataFrame MC

In [ ]:
from cols_to_keep import *

#Load CV dataframe
keys2load = ["cc1pi", "hdr", "histpotdf", "nudf"] ## keys from the configuration file

bnb_path = "/exp/sbnd/data/users/lpelegri/cafpyana_data/mc_ar23p_extended_syst_pruned.df"
mc_bnb_df = load_df(bnb_path, keys2load, 100, reprocess_df = False, reprocess_truth = False)
mc_bnb_evt_df = mc_bnb_df['cc1pi']
mc_bnb_nu_df = mc_bnb_df['nudf']
mc_bnb_hdr_df = mc_bnb_df['hdr']

reco_cols_to_keep =  min_reco_cols_to_keep + [
    ('truth','nu_categ', '', '', '',''),
    ('truth','genie_categ', '', '', '',''),
    ('truth','nu_categ_proton_reduced', '', '', '','')
]

mc_bnb_evt_df[reco_cols_to_keep]

#Load data
keys2load = ["cc1pi_good", "hdr", "histpotdf"] ## keys from the configuration file
data_df = load_df("/exp/sbnd/data/users/lpelegri/cafpyana_data/cc1pi_data_fixdev_bnblight_quality_cut.df", keys2load, 100)
data_evt_df = data_df['cc1pi_good']
data_hdr_df = data_df['hdr']
del data_df
gc.collect()

keys2load = ["cc1pi", "hdr", "histpotdf"] ## keys from the configuration file
#load off beam light df
off_beam_light_df = load_df("/exp/sbnd/data/users/lpelegri/cafpyana_data/cc1pi_data_offbeamlight.df", keys2load, 100)
off_beam_light_evt_df = off_beam_light_df['cc1pi']
off_beam_light_hdr_df = off_beam_light_df['hdr']
del off_beam_light_df
gc.collect()

In [ ]:
pot_weight_col = ('slc', 'wgt', '', '', '', '')

# BNB data
print("data_tot_pot: %.3e" %(data_tot_pot))
print("data tot gates : %.3e" %(data_gates))
data_evt_df[pot_weight_col] = np.ones(len(data_evt_df))

# BNB MC
mc_tot_pot = mc_bnb_hdr_df['pot'].sum()
print("mc_tot_pot: %.3e" %(mc_tot_pot))
mc_pot_scale = data_tot_pot / mc_tot_pot
print("mc_pot_scale: %.3e" %(mc_pot_scale))
mc_bnb_evt_df[pot_weight_col] = mc_pot_scale * np.ones(len(mc_bnb_evt_df))

off_beam_data_gates = off_beam_light_hdr_df.noffbeambnb.sum()
print("intime cosmics data gates: {:.2e}".format(off_beam_data_gates))
f = 0.075
scale_off_beam_to_lightdata = (1-f)*data_gates/off_beam_data_gates
print("goal scale: {:.2f}".format(scale_off_beam_to_lightdata))
off_beam_light_evt_df[pot_weight_col] = scale_off_beam_to_lightdata * np.ones(len(off_beam_light_evt_df))



In [ ]:
#Add MC stat:
mc_evt_df = mc_bnb_evt_df

# Test background composition

In [ ]:
mc_cumulative_masks = build_event_cumulative_masks(mc_evt_df, sideband = "")
data_cumulative_masks = build_event_cumulative_masks(data_evt_df, sideband = "")

In [ ]:
for name in mc_cumulative_masks.keys():
    n_mc   = get_n_evt(mc_evt_df,   mc_cumulative_masks[name],   use_weight=True)
    n_data = get_n_evt(data_evt_df, data_cumulative_masks[name], use_weight=False)
    print(f"{name:<15} | {n_mc:<12.2f} | {n_data:<12}")

In [ ]:
for key in ["0p", "1p", "2plusp"]:
    mask = data_cumulative_masks["energy"] & mask_dict[key](data_evt_df)
    print(f"{key}: {get_n_evt(data_evt_df, use_weight=False, mask=mask)}")

# Version that imports the systematics for the final vars

In [ ]:
def filter_df(df, mask_key, cut_mask, config):
    """Applies cut masks, extra masks, and slice grouping."""
    filtered = df[cut_mask & mask_dict[config.extra_mask](df)]
    if config.first_per_slice:
        filtered = filtered.groupby(level=SLICE_LEVELS, sort=False).first()
    return filtered

In [ ]:
final_var_configs = [config_all_evts_final, config_p_mu_final, config_cos_theta_mu_final, config_p_pi_final,
                     config_cos_theta_pi_final, config_delta_alpha_T_final, config_delta_pT_final,
                    config_delta_phi_T_final, config_num_protons,config_angle_between_candidates_final]

In [ ]:
import os, shutil, tarfile
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd


config_p_pi_final_p_type = FullHistogramConfig(
    file_name = "pion_p_p_type",      
    var_evt_reco_col=('slc','measure_var','TLE_p_pi','','',''),
    truth_column = pion_p_type_column,
    first_per_slice = True,
    start_cut = "energy",
    end_cut = "energy",
    bins= np.array([0.13, 0.218, 0.296,0.415,2]),
    xlabel=r"$P_\pi$ [GeV]",
    ylabel=slices_y_label
)


# --- Configuration & Setup ---
config_vec =  [config_p_pi_final_p_type] + final_var_configs
FULL_SYST = True

if FULL_SYST:
    parent_path = "/exp/sbnd/data/users/lpelegri/Graphs/FinalMCDataCompGraphsFullErr"
else:
    parent_path = "/exp/sbnd/data/users/lpelegri/Graphs/FinalMCDataCompGraphs"

'''
if plot_sideband:
    parent_path += "sideband"
'''

good_base_path, bad_base_path = os.path.join(parent_path, "good"), os.path.join(parent_path, "bad")
tar_output_path = f"{parent_path}.tar"

CHI2_THRESHOLD = 5
PLOT_IND_UNCR = True
WGT_COL = ('slc', 'wgt', '', '', '', '')
SLICE_LEVELS = ['__ntuple', 'entry', 'rec.slc..index']

# Color codes
RED, RESET = "\033[31m", "\033[0m"

# Clean workspace
for path in [parent_path, tar_output_path]:
    if os.path.exists(path):
        shutil.rmtree(path) if os.path.isdir(path) else os.remove(path)
os.makedirs(good_base_path, exist_ok=True); os.makedirs(bad_base_path, exist_ok=True)

file_dir = "/exp/sbnd/data/users/lpelegri/syst/frac_cov_matrices_rate_no_bkg_substract"

# Load Systematic Dictionaries
flux_syst = np.load(file_dir + "/extended_flux_syst_dict_rate_ar23p.npz")
g4_syst = np.load(file_dir + "/extended_g4_syst_dict_rate_ar23p.npz")
genie_syst = np.load(file_dir + "/extended_genie_syst_dict_xsec_ar23p.npz")
detvar_syst = np.load(file_dir + "/detvar_syst_dict.npz")
mcstat_syst = np.load(file_dir + "/mcstat_syst_dict_ar23p.npz")


file_dir = "/exp/sbnd/data/users/lpelegri/syst/frac_cov_matrices"
cosmics_syst = np.load(file_dir + "/cosmics_syst_dict.npz")

'''
flux_syst = np.load(file_dir + "/extended_flux_syst_dict.npz")
g4_syst = np.load(file_dir + "/extended_g4_syst_dict.npz")
genie_syst = np.load(file_dir + "/extended_genie_xsec_syst_dict.npz")
mcstat_syst = np.load(file_dir + "/mcstat_syst_dict.npz")
cosmics_syst = np.load(file_dir + "/cosmics_syst_dict.npz")
detvar_syst = np.load(file_dir + "/detvar_syst_dict.npz")
'''
pot_frac_unc = 0.02
ntargets_frac_unc = 0.01


GENIE_TAGS = [False, True]

# --- Main Analysis Loop ---
for PLOT_GENIE_CATEG in GENIE_TAGS:
    for config in config_vec:
        # Determine cut range
        idx_range = sorted([cuts.index(config.start_cut), cuts.index(config.end_cut)])
        selected_cuts = cuts[idx_range[0] : idx_range[1] + 1]
    
        if config.file_name != "pion_p_p_type":
            if PLOT_GENIE_CATEG:
                config.truth_column = ('truth','genie_categ','','','','')
            else:
                config.truth_column = ('truth','nu_categ','','','','')
            
        for cut in selected_cuts: 
            print(f"--- Processing Cut: {cut} ---")
            
            # 1. Prepare Dataframes
            curr_mc = filter_df(mc_evt_df, config.extra_mask, mc_cumulative_masks[cut], config)
            curr_data = filter_df(data_evt_df, config.extra_mask, data_cumulative_masks[cut], config)
            
            # Determine number of bins from config to initialize matrices
            # (This avoids the IndexError by matching the dimension of the loaded matrices)
            n_bins = len(config.bins) - 1
            bin_centers = (config.bins[:-1] + config.bins[1:]) / 2.
    
            # 2. Systematic Matrix Aggregation
            # We start with MC Statistics as the baseline matrix
            
            if config.file_name == "pion_p_p_type":
                total_cov_frac = mcstat_syst["pion_p"].copy()
            else:
                total_cov_frac = mcstat_syst[config.file_name].copy()
            frac_unc_list = [(np.sqrt(np.diag(total_cov_frac)), "MCStat")]
            
            if FULL_SYST:
                # Binned Systematics (Matrices)
                systs = [flux_syst, g4_syst, genie_syst, detvar_syst, cosmics_syst]
                syst_names = ["Flux", "G4", "Genie", "Detector", "Cosmic"]
    
                for name, syst_dict in zip(syst_names, systs):
                    if config.file_name == "pion_p_p_type":
                        matrix = syst_dict["pion_p"]
                    else:
                        matrix = syst_dict[config.file_name]
                        
                    total_cov_frac += matrix # Matrix addition preserves correlations
                    frac_unc_list.append((np.sqrt(np.diag(matrix)), name))
    
                # Flat Systematics (Normalization)f
                # These are 100% correlated across all bins
                flat_systs = [pot_frac_unc, ntargets_frac_unc]
                flat_names = ["POT", "Ntargets"]
    
                for name, val in zip(flat_names, flat_systs):
                    # Create a matrix where every element is (sigma_flat)^2
                    flat_matrix = np.full((n_bins, n_bins), val**2)
                    total_cov_frac += flat_matrix
                    frac_unc_list.append((val * np.ones(n_bins), name))
    
            # 3. Final Uncertainty Vector for Plotting
            total_uncertainty_vec = np.sqrt(np.diag(total_cov_frac))
            final_plot_list = [(total_uncertainty_vec, "Total")] + frac_unc_list
            plot_frac_unc(final_plot_list, config)
            
            # 4. Convert Fractional Covariance to Absolute Covariance for Chi2
            # Use the MC counts for the current cut to scale the matrix
            # 'ret' logic usually comes from the plotter; ensure you have mc_counts here
            mc_counts, _ = np.histogram(curr_mc[config.var_evt_reco_col], bins=config.bins, weights=curr_mc[WGT_COL])
            
            total_cov = cov_from_fraccov(total_cov_frac, mc_counts)
    
            # 5. Plotting and Chi2 Calculation
            fig, red_chi2 = plot_stacked_histogram_with_ratio(
                mc_df=curr_mc, data_df=curr_data, config=config,
                cov_frac_matrix=total_cov_frac, cov_matrix=total_cov,
                title=f"{cut} - {config.file_name}", weight_column=WGT_COL,
                data_pot=data_tot_pot, normalize=False, show_stats=FULL_SYST, symmetric_ratio = True, divide_by_bin_width = True
            )
    
            # 6. File Management
            status_path = good_base_path if red_chi2 < CHI2_THRESHOLD else bad_base_path
            save_dir = os.path.join(status_path, config.file_name)
            os.makedirs(save_dir, exist_ok=True)

            topology_name = "topology"
            if PLOT_GENIE_CATEG:
                topology_name = "genie"
            f_name = f"cut_{cut}_{config.file_name}_{topology_name}.pdf"
            if config.file_name == "pion_p_p_type":
                f_name = f"cut_{cut}_{config.file_name}.pdf"
            
            fig.savefig(os.path.join(save_dir, f_name), format='pdf', bbox_inches='tight')
            plt.close(fig)
    
            color = RED if red_chi2 > CHI2_THRESHOLD else ""
            print(f"{color}Saved {f_name} (Chi2: {red_chi2:.2f}){RESET}")

# --- Archive Results ---
print(f"Archiving to {tar_output_path}...")
with tarfile.open(tar_output_path, "w:gz") as tar:
    tar.add(parent_path, arcname=os.path.basename(parent_path))
print("Done!")